# Strait of Hormuz 2026 — Maritime Trade & Risk Analytics

This notebook implements the coding pipeline on the supplied simulation dataset.

**Important:** this is a simulation-based dataset. The analysis describes patterns in the supplied data and does not independently validate its geopolitical claims.

## 1. Setup and load data

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

ROOT = Path('..')
DATA = ROOT / 'data' / 'hormuz_trade_tier_continental_2026.csv'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['reroute_flag'] = (df['rerouted'] == 'Yes (Cape of Good Hope)').astype(int)
print(df.shape)
df.head()

## 2. Data quality and consistency checks

In [ ]:
quality = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'missing': [df[c].isna().sum() for c in df.columns],
    'missing_pct': [df[c].isna().mean()*100 for c in df.columns],
    'unique': [df[c].nunique(dropna=True) for c in df.columns],
})
display(quality)

df['asset_value_check_diff'] = df['total_asset_value_at_risk_usd'] - df['ship_hull_value_usd'] - df['estimated_cargo_value_usd']
df['transit_cost_check_diff'] = df['total_transit_cost_usd'] - df['toll_usd'] - df['insurance_cost_usd'] - df['reroute_penalty_usd']
print('Asset formula mismatches:', (df['asset_value_check_diff'] != 0).sum())
print('Transit-cost formula mismatches:', (df['transit_cost_check_diff'] != 0).sum())

## 3. Descriptive analytics

In [ ]:
tier_summary = df.groupby('trade_tier').agg(
    records=('mmsi','size'), unique_vessels=('mmsi','nunique'), reroute_rate=('reroute_flag','mean'),
    avg_delay_days=('days_delayed','mean'), avg_transit_cost_usd=('total_transit_cost_usd','mean'),
    total_transit_cost_usd=('total_transit_cost_usd','sum'), total_asset_risk_usd=('total_asset_value_at_risk_usd','sum'),
    avg_insurance_cost_usd=('insurance_cost_usd','mean'), avg_extra_fuel_tonnes=('extra_fuel_tonnes','mean')
).reset_index()
tier_summary['reroute_rate_pct'] = tier_summary['reroute_rate']*100
display(tier_summary)

display(pd.crosstab(df['trade_tier'], df['reroute_flag'], normalize='index').round(3))

## 4. Statistical testing

In [ ]:
taxed = df.loc[df.trade_tier=='Taxed','total_transit_cost_usd']
priv = df.loc[df.trade_tier=='Privileged','total_transit_cost_usd']
tt = ttest_ind(taxed, priv, equal_var=False)
mw = mannwhitneyu(taxed, priv, alternative='two-sided')
ct = pd.crosstab(df['trade_tier'], df['reroute_flag'])
chi2, chi_p, chi_dof, expected = chi2_contingency(ct)
print('Welch t-test p-value:', tt.pvalue)
print('Mann-Whitney p-value:', mw.pvalue)
print('Chi-square p-value:', chi_p)

## 5. Daily operational trends and anomalies

In [ ]:
daily = df.groupby('date').agg(
    vessel_records=('mmsi','size'), unique_vessels=('mmsi','nunique'), reroute_rate=('reroute_flag','mean'),
    avg_delay_days=('days_delayed','mean'), total_transit_cost_usd=('total_transit_cost_usd','sum'),
    total_asset_risk_usd=('total_asset_value_at_risk_usd','sum'), total_fuel_tonnes=('extra_fuel_tonnes','sum')
).reset_index().sort_values('date')
daily['reroute_rate_pct'] = daily['reroute_rate']*100
daily['rolling_7d_cost_usd'] = daily['total_transit_cost_usd'].rolling(7,min_periods=3).mean()
rm = daily['total_transit_cost_usd'].rolling(30,min_periods=10).mean()
rs = daily['total_transit_cost_usd'].rolling(30,min_periods=10).std()
daily['cost_z_30d'] = (daily['total_transit_cost_usd']-rm)/rs
daily['cost_anomaly'] = daily['cost_z_30d'].abs()>3
display(daily.head())

plt.figure(figsize=(11,5))
plt.plot(daily.date, daily.total_transit_cost_usd, label='Daily total transit cost')
plt.plot(daily.date, daily.rolling_7d_cost_usd, label='7-day rolling average')
plt.title('Daily Transit Cost Through the Simulated Hormuz Crisis')
plt.xlabel('Date'); plt.ylabel('USD'); plt.legend(); plt.tight_layout(); plt.show()

## 6. Rerouting classification

The target is `reroute_flag`. `transit_status` and outcome-derived variables are excluded to prevent target leakage.

In [ ]:
features = ['trade_tier','flag','commodity','destination','payment_rail','continent',
            'ship_hull_value_usd','estimated_cargo_value_usd','insurance_premium_delta_pct']
X = df[features]; y = df['reroute_flag']
dates = sorted(df.date.dropna().unique())
cutoff = dates[int(len(dates)*0.70)]
train_mask = df.date < cutoff
test_mask = ~train_mask
X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
cat = [c for c in features if X[c].dtype=='object']
num = [c for c in features if c not in cat]
pre = ColumnTransformer([
    ('cat', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), cat),
    ('num', SimpleImputer(strategy='median'), num)
])
lr = Pipeline([('preprocess',pre),('model',LogisticRegression(max_iter=2000,class_weight='balanced'))])
lr.fit(X_train,y_train)
lr_prob = lr.predict_proba(X_test)[:,1]; lr_pred=(lr_prob>=0.5).astype(int)
Xtr = pre.fit_transform(X_train); Xte = pre.transform(X_test)
xgb = XGBClassifier(n_estimators=250,max_depth=4,learning_rate=.05,subsample=.8,colsample_bytree=.8,reg_lambda=1,eval_metric='logloss',random_state=42,n_jobs=2)
xgb.fit(Xtr,y_train)
xgb_prob=xgb.predict_proba(Xte)[:,1]; xgb_pred=(xgb_prob>=0.5).astype(int)
results = pd.DataFrame([
    ['Logistic Regression',accuracy_score(y_test,lr_pred),precision_score(y_test,lr_pred,zero_division=0),recall_score(y_test,lr_pred,zero_division=0),f1_score(y_test,lr_pred,zero_division=0),roc_auc_score(y_test,lr_prob)],
    ['XGBoost',accuracy_score(y_test,xgb_pred),precision_score(y_test,xgb_pred,zero_division=0),recall_score(y_test,xgb_pred,zero_division=0),f1_score(y_test,xgb_pred,zero_division=0),roc_auc_score(y_test,xgb_prob)]
], columns=['model','accuracy','precision','recall','f1','roc_auc'])
display(results)

### Model interpretation note

If XGBoost reaches near-perfect performance, do not present that as proof of real-world predictability. Synthetic datasets often contain deterministic rules. Inspect feature importance and cross-check whether categorical variables encode the target.

In [ ]:
feature_names = pre.get_feature_names_out()
importance = pd.DataFrame({'feature':feature_names,'importance':xgb.feature_importances_}).sort_values('importance',ascending=False)
display(importance.head(15))

## 7. Isolation Forest anomaly detection

In [ ]:
anomaly_features=['vessel_records','reroute_rate_pct','avg_delay_days','total_transit_cost_usd','total_asset_risk_usd','total_fuel_tonnes']
A=daily[anomaly_features].replace([np.inf,-np.inf],np.nan).fillna(0)
iso=IsolationForest(n_estimators=300,contamination=.05,random_state=42)
daily['isolation_forest_label']=iso.fit_predict(A)
daily['isolation_forest_anomaly']=daily['isolation_forest_label'].eq(-1)
display(daily.loc[daily.isolation_forest_anomaly,['date']+anomaly_features])

## 8. Export Power BI-ready tables

In [ ]:
tier_summary.to_csv(OUT/'trade_tier_summary.csv',index=False)
daily.to_csv(OUT/'daily_operational_metrics_with_anomalies.csv',index=False)
results.to_csv(OUT/'model_comparison.csv',index=False)
importance.to_csv(OUT/'xgb_feature_importance.csv',index=False)
print('Saved to', OUT.resolve())